In [ ]:
# Week 3 Day 3 — Neural Network Basics
# Goal: Build neural network from scratch

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [ ]:
# Cell 2 — Single neuron calculation
# Manual calculation — no PyTorch magic yet

print("── Single Neuron ──")

# One CT voxel as input
hu_value = torch.tensor(200.0)
print(f"Input HU value: {hu_value}")

# Learnable parameters
weight = torch.tensor(0.4)
bias   = torch.tensor(0.1)
print(f"Weight: {weight}")
print(f"Bias:   {bias}")

# Neuron calculation
output = (hu_value * weight) + bias
print(f"\nCalculation: ({hu_value} × {weight}) + {bias}")
print(f"Output: {output}")

print("\n── Multiple inputs ──")
# Three CT voxels as input
hu_values = torch.tensor([200.0, 50.0, -500.0])
weights   = torch.tensor([0.4,   0.3,   0.2])
bias      = torch.tensor(0.1)

# Multiply each input by its weight
weighted  = hu_values * weights
print(f"Inputs:   {hu_values}")
print(f"Weights:  {weights}")
print(f"Weighted: {weighted}")

# Sum all weighted inputs + bias
output = weighted.sum() + bias
print(f"Sum + bias: {output}")

── Single Neuron ──
Input HU value: 200.0
Weight: 0.4000000059604645
Bias:   0.10000000149011612

Calculation: (200.0 × 0.4000000059604645) + 0.10000000149011612
Output: 80.0999984741211

── Multiple inputs ──
Inputs:   tensor([ 200.,   50., -500.])
Weights:  tensor([0.4000, 0.3000, 0.2000])
Weighted: tensor([  80.0000,   15.0000, -100.0000])
Sum + bias: -4.900000095367432


In [ ]:
# Cell 3 — ReLU activation

print("── ReLU function ──")
print("Positive → unchanged")
print("Negative → zero")

# Create test values
test_values = torch.tensor([-5.0, -2.0, -0.5,
                              0.0,
                              0.5,  2.0,  5.0])

# Apply ReLU
relu = nn.ReLU()
output = relu(test_values)

print(f"\nInput:  {test_values}")
print(f"Output: {output}")

print("\n── ReLU on neuron output ──")
# From Cell 2 our neuron output was -4.9
neuron_output = torch.tensor(-4.9)
after_relu = relu(neuron_output)
print(f"Neuron output: {neuron_output}")
print(f"After ReLU:    {after_relu}")
print("→ Neuron stays silent")

print("\n── ReLU on CT feature map ──")
# Simulate small feature map after convolution
feature_map = torch.tensor([[ 3.2, -1.5,  2.1],
                             [-0.8,  4.5, -2.3],
                             [ 1.1, -3.2,  0.7]])

print(f"Feature map before ReLU:\n{feature_map}")
after_relu = relu(feature_map)
print(f"\nFeature map after ReLU:\n{after_relu}")
print("\nNegative values → 0 (silent)")
print("Positive values → kept (active)")

── ReLU function ──
Positive → unchanged
Negative → zero

Input:  tensor([-5.0000, -2.0000, -0.5000,  0.0000,  0.5000,  2.0000,  5.0000])
Output: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 2.0000, 5.0000])

── ReLU on neuron output ──
Neuron output: -4.900000095367432
After ReLU:    0.0
→ Neuron stays silent

── ReLU on CT feature map ──
Feature map before ReLU:
tensor([[ 3.2000, -1.5000,  2.1000],
        [-0.8000,  4.5000, -2.3000],
        [ 1.1000, -3.2000,  0.7000]])

Feature map after ReLU:
tensor([[3.2000, 0.0000, 2.1000],
        [0.0000, 4.5000, 0.0000],
        [1.1000, 0.0000, 0.7000]])

Negative values → 0 (silent)
Positive values → kept (active)


In [ ]:
# Cell 4 — Sigmoid activation

print("── Sigmoid function ──")
print("Converts any number to 0-1 range")
print("Used in final layer for probability")

# Sigmoid reference values
test_values = torch.tensor([-5.0, -3.0, -2.0,
                             -1.0,  0.0,  1.0,
                              2.0,  3.0,  5.0])

sigmoid = nn.Sigmoid()
output = sigmoid(test_values)

print(f"\nInput  → Output")
for i, o in zip(test_values, output):
    print(f"{i:6.1f} → {o:.4f}  ({o*100:.1f}%)")

print("\n── Sigmoid on neuron output ──")
# Three different neuron outputs
outputs = torch.tensor([-3.0, 0.0, 3.0])
probs = sigmoid(outputs)

print(f"\nNeuron output: {outputs}")
print(f"Probability:   {probs}")
print(f"\n-3.0 → {probs[0]:.4f} → {probs[0]*100:.1f}% tumour probability")
print(f" 0.0 → {probs[1]:.4f} → {probs[1]*100:.1f}% tumour probability")
print(f" 3.0 → {probs[2]:.4f} → {probs[2]*100:.1f}% tumour probability")

── Sigmoid function ──
Converts any number to 0-1 range
Used in final layer for probability

Input  → Output
  -5.0 → 0.0067  (0.7%)
  -3.0 → 0.0474  (4.7%)
  -2.0 → 0.1192  (11.9%)
  -1.0 → 0.2689  (26.9%)
   0.0 → 0.5000  (50.0%)
   1.0 → 0.7311  (73.1%)
   2.0 → 0.8808  (88.1%)
   3.0 → 0.9526  (95.3%)
   5.0 → 0.9933  (99.3%)

── Sigmoid on neuron output ──

Neuron output: tensor([-3.,  0.,  3.])
Probability:   tensor([0.0474, 0.5000, 0.9526])

-3.0 → 0.0474 → 4.7% tumour probability
 0.0 → 0.5000 → 50.0% tumour probability
 3.0 → 0.9526 → 95.3% tumour probability


In [ ]:
# Cell 5 — Simple neural network
# Two layers — forward pass

print("── Building simple network ──")

# Define network
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 1: 1 input → 4 neurons
        self.layer1 = nn.Linear(1, 4)
        self.relu   = nn.ReLU()
        # Layer 2: 4 inputs → 1 output
        self.layer2 = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Step 1 → layer 1
        x = self.layer1(x)
        print(f"After layer 1: {x.shape}")
        # Step 2 → ReLU
        x = self.relu(x)
        print(f"After ReLU:    {x.shape}")
        # Step 3 → layer 2
        x = self.layer2(x)
        print(f"After layer 2: {x.shape}")
        # Step 4 → Sigmoid
        x = self.sigmoid(x)
        print(f"After Sigmoid: {x.shape}")
        return x

# Create network
model = SimpleNet().to(device)
print(f"Network created on: {device}")

print("\n── Forward pass ──")
# Single HU value as input
hu_input = torch.tensor([[200.0]]).to(device)
print(f"Input: {hu_input}  shape: {hu_input.shape}")

print("\nPassing through network:")
output = model(hu_input)
print(f"\nFinal output: {output.item():.4f}")
print(f"Tumour probability: {output.item()*100:.1f}%")


── Building simple network ──
Network created on: cuda

── Forward pass ──
Input: tensor([[200.]], device='cuda:0')  shape: torch.Size([1, 1])

Passing through network:
After layer 1: torch.Size([1, 4])
After ReLU:    torch.Size([1, 4])
After layer 2: torch.Size([1, 1])
After Sigmoid: torch.Size([1, 1])

Final output: 0.0000
Tumour probability: 0.0%


In [ ]:
# Cell 6 — Loss function
# Measures how wrong prediction is
# Cell 6 — Loss function
import torch
import torch.nn as nn

print("── Binary Cross Entropy Loss ──")

# Setup
bce_loss = nn.BCELoss()
sigmoid  = nn.Sigmoid()

# Three scenarios

print("\nScenario 1 — Perfect prediction:")
prediction = torch.tensor([0.99])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\nScenario 2 — Wrong prediction:")
prediction = torch.tensor([0.01])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\nScenario 3 — Uncertain prediction:")
prediction = torch.tensor([0.50])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\n── Loss summary ──")
print("Perfect prediction   → loss close to 0")
print("Wrong prediction     → loss very large")
print("Uncertain prediction → loss moderate")

── Binary Cross Entropy Loss ──

Scenario 1 — Perfect prediction:
Prediction: 0.99  Label: 1  Loss: 0.0101

Scenario 2 — Wrong prediction:
Prediction: 0.01  Label: 1  Loss: 4.6052

Scenario 3 — Uncertain prediction:
Prediction: 0.50  Label: 1  Loss: 0.6931

── Loss summary ──
Perfect prediction   → loss close to 0
Wrong prediction     → loss very large
Uncertain prediction → loss moderate


In [ ]:
# Cell 6 — Loss function
import torch
import torch.nn as nn

print("── Binary Cross Entropy Loss ──")

bce_loss = nn.BCELoss()
sigmoid  = nn.Sigmoid()

print("\nScenario 1 — Perfect prediction:")
prediction = torch.tensor([0.99])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\nScenario 2 — Wrong prediction:")
prediction = torch.tensor([0.01])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\nScenario 3 — Uncertain prediction:")
prediction = torch.tensor([0.50])
label      = torch.tensor([1.0])
loss = bce_loss(prediction, label)
print(f"Prediction: {prediction.item():.2f}  Label: {label.item():.0f}  Loss: {loss.item():.4f}")

print("\n── Loss summary ──")
print("Perfect prediction   → loss close to 0")
print("Wrong prediction     → loss very large")
print("Uncertain prediction → loss moderate")

── Binary Cross Entropy Loss ──

Scenario 1 — Perfect prediction:
Prediction: 0.99  Label: 1  Loss: 0.0101

Scenario 2 — Wrong prediction:
Prediction: 0.01  Label: 1  Loss: 4.6052

Scenario 3 — Uncertain prediction:
Prediction: 0.50  Label: 1  Loss: 0.6931

── Loss summary ──
Perfect prediction   → loss close to 0
Wrong prediction     → loss very large
Uncertain prediction → loss moderate
